# Phase 2: Data Preprocessing & Reconciliation Check
## Project: Workforce Attrition Patterns and Risk Hotspot Analysis
**Scope:** Phase 2 – Verification of Analysis-Ready Dataset & Feature Derivations

> **DISCLAIMER:** The dataset provenance is UNVERIFIED. This data is analyzed as a public/synthetic HR benchmark dataset. Findings must not be described as real Palo Alto Networks employment records or proprietary corporate data. No modeling or EDA conclusions are drawn in this verification notebook.

In [ ]:
import sys
from pathlib import Path

# Ensure project src is in python path
project_root = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
sys.path.insert(0, str(project_root / 'src'))

import pandas as pd
from IPython.display import display

from workforce_intel.config import (
    PROCESSED_DATA_FILE, PROCESSED_SCHEMA_FILE, RAW_DATA_FILE
)
from workforce_intel.io import load_processed, load_raw
from workforce_intel.validation import reconciliation_audit

print('Libraries and verification modules imported successfully.')

## 1. Load Datasets & Execute Reconciliation
Verify row-level and cell-for-cell identity between raw and processed datasets.

In [ ]:
raw_df, delimiter = load_raw(RAW_DATA_FILE)
processed_df = load_processed(PROCESSED_DATA_FILE, PROCESSED_SCHEMA_FILE)

print(f'Raw Shape: {raw_df.shape}')
print(f'Processed Shape: {processed_df.shape}')

recon = reconciliation_audit(raw_df, processed_df)
display(recon['reconciliation_table'])

## 2. Derived Feature Specifications
Audit the 15 derived features, non-null counts, and value domains.

In [ ]:
display(recon['derived_feature_audit'])

## 3. Subgroup & Bucket Sizes Audit (n < 30 Rule)
Inspect distribution counts and flags for all derived categorical bands and indicators.

In [ ]:
bucket_df = recon['bucket_sizes_table']
display(bucket_df)

small_groups = bucket_df[bucket_df['is_small_group']]
print(f'Small subgroups (n < 30) detected: {len(small_groups)}')

## 4. Zero-Denominator Rule & Missingness Audit
Verify that only RoleTenureShare and PromotionGapShare contain NaNs (exactly 44 rows where YearsAtCompany == 0).

In [ ]:
nan_df = recon['nan_audit_table']
display(nan_df[nan_df['nan_count'] > 0])

assert (nan_df.loc[~nan_df['column'].isin(['RoleTenureShare', 'PromotionGapShare']), 'nan_count'] == 0).all()
print('Zero unexpected NaNs across all other 45 columns verified.')